# Flow Matching on $\mathbb{R}^2$
In this notebook, we perform flow matching on the two dimensional translation group $\mathbb{R}^2$, which is isometric to two dimensional Euclidean space.

In [ ]:
import torch
import numpy as np
from lieflow.groups import Rn
from lieflow.models import (
    get_model_FM,
    LogarithmicDistance,
    DataGenerator,
)
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from matplotlib.cm import ScalarMappable
from tqdm.notebook import tqdm
from functools import partial
import json
with open("plot.json", "r") as f:
    plot_config = json.load(f)
latex_line_width = plot_config.pop("latex_line_width")
plt.rcParams.update(plot_config)

device = "cpu"

In [ ]:
r2 = Rn(2)

In [ ]:
generate_videos = True

# Distributions

In [ ]:
def generate_normals(N):
    return torch.randn(N, 2, device=device)

def generate_uniforms_on_circle(N, ε=0.04):
    normals = generate_normals(N)
    lengths = (normals**2).sum(dim=-1).sqrt()
    non_zero = lengths > 0.
    circle_samples = normals[non_zero] / lengths[non_zero, None]
    return circle_samples + ε * torch.randn(circle_samples.shape, device=device)

def plot_x(ax, x, **kwargs):
    x = x.to("cpu")
    return ax.scatter(x[..., 0], x[..., 1], **kwargs)

In [ ]:
tests = ("normals_to_circle",)

In [ ]:
def data_generator(test, ε=0.04):
    match test:
        case "normals_to_circle":
            generate_x_0 = generate_normals
            generate_x_1 = partial(generate_uniforms_on_circle, ε=ε)
    return generate_x_0, generate_x_1

# Models

## Training

In [ ]:
with open("config.json", "r") as f:
    config_space = json.load(f)["R2"]
# Parameters shared within the space, which may be overridden per test case.
config_shared = {key: value for key, value in config_space.items() if not isinstance(value, dict)}
config = {test: config_shared | value for test, value in config_space.items() if isinstance(value, dict)}

In [ ]:
def train_model(generate_x_0, generate_x_1, config):
    train_loader = DataGenerator(generate_x_0, generate_x_1, config["batch_size"], config["batches_per_epoch"])

    model_FM = get_model_FM(r2, H=config["width"], L=config["depth"], shared_weights=config["shared_weights"]).to(device)
    print("Number of parameters: ", model_FM.parameter_count)
    optimizer_FM = torch.optim.Adam(model_FM.parameters(), config["learning_rate"], weight_decay=config["weight_decay"])
    loss = LogarithmicDistance(torch.tensor(config["loss_weights"], device=device))

    losses_FM = np.zeros(config["epochs"])
    for i in tqdm(range(config["epochs"])):
        losses_FM[i] = model_FM.train_network(device, train_loader, optimizer_FM, loss, show_progress=False)

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.plot(losses_FM)
    ax.set_title("Batch Loss Flow Matching")
    ax.set_xscale("log")
    ax.set_yscale("log")

    return model_FM

In [ ]:
models_FM = {}
for test in tests:
    print(test)
    generate_x_0, generate_x_1 = data_generator(test, ε=config[test]["epsilon"])

    x_0s = generate_x_0(32)
    x_1s = generate_x_1(32)

    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    plot_x(ax[0], x_0s[:32])
    ax[0].set_xlim(-3, 3)
    ax[0].set_ylim(-3, 3)
    ax[0].set_title(r"$\mathfrak{X}_0$")
    plot_x(ax[1], x_1s[:32])
    ax[1].set_xlim(-3, 3)
    ax[1].set_ylim(-3, 3)
    ax[1].set_title(r"$\mathfrak{X}_1$")

    models_FM[test] = train_model(generate_x_0, generate_x_1, config[test])

# Testing

In [ ]:
def create_animations(tests, models_FM, N_samples=2**5, N_steps=120):
    N_models = len(tests)

    x_0s_test = {}
    x_ts_test = {}
    for test in tests:
        generate_x_0, _ = data_generator(test, ε=config[test]["epsilon"])
        x_0s_test[test] = generate_x_0(N_samples).to(device)
        x_ts_test[test] = x_0s_test[test].detach().clone()
        
        models_FM[test].eval()

    t = 0
    Δt = 1. / N_steps

    for test in tests:
        metadata = {'title': 'Flow Matching R^2', 'artist': 'Matplotlib'}
        writer = FFMpegWriter(fps=30, metadata=metadata)

        fig, ax = plt.subplots(1, N_models, figsize=(5 * N_models, 5))
        ax.set_xlim(-3, 3)
        ax.set_ylim(-3, 3)

        with writer.saving(fig, f"output/flow_matching_R2_{test}.mp4", dpi=150):
            scatter = ax.scatter([], [])
            for frame in tqdm(range(N_steps + 1)):
                scatter.remove()
                scatter = plot_x(ax, x_ts_test[test], color="tab:blue")
                ax.set_title(fr"$\mathfrak{{X}}_{{{t:.2f}}}$")
                writer.grab_frame()
                t = frame * Δt
                with torch.no_grad():
                    x_ts_test[test] = models_FM[test].step(x_ts_test[test], torch.tensor([t], device=device), Δt)

In [ ]:
if generate_videos:
    create_animations(tests, models_FM, N_samples=2**5)

In [ ]:
t = 0
N_steps = config_shared["N_steps"]
Δt = 1. / N_steps
N_show = 5
N_skip = int(N_steps / (N_show-1))

N_models = len(tests)
N_samples = 32

x_0s_test = {}
x_1s_test = {}
x_ts_test = {}
for test in tests:
    generate_x_0, generate_x_1 = data_generator(test, ε=config[test]["epsilon"])
    x_0s_test[test] = generate_x_0(N_samples).to(device)
    x_1s_test[test] = generate_x_1(N_samples).to(device)
    x_ts_test[test] = x_0s_test[test].detach().clone()
    
    models_FM[test].eval()

fig = plt.figure(figsize=(4.8, 1.6 * N_models * 3/3.1))
gs = gridspec.GridSpec(N_models, 4, width_ratios=[1, 1, 1, 0.1], height_ratios=N_models * [1.], wspace=0.1, hspace=0.1)
cax = fig.add_subplot(gs[:, 3])

ax = []
for i in range(N_models):
    ax.append([])
    for j in range(3):
        a = fig.add_subplot(gs[i, j])
        a.set_xlim(-3, 3)
        a.set_ylim(-3, 3)
        a.set_xticks([])
        a.set_yticks([])
        a.set_aspect("equal")
        ax[i].append(a)
        
ax[0][0].set_title(r"$\mathfrak{X}_0$")
ax[0][1].set_title(r"$\mathfrak{X}_t$")
ax[0][2].set_title(r"$\mathfrak{X}_1$")
Δc = 1 / (N_show - 1)
colors = [(j * Δc, 0.1, 1 - j * Δc) for j in range(N_show)]
cmap = mcolors.ListedColormap(colors)
for i, test in enumerate(tests):
    k = 0

    plot_x(ax[i][0], x_0s_test[test][:N_samples], marker=".")
    plot_x(ax[i][2], x_1s_test[test][:N_samples], marker=".")

    alpha = 1
    N_samples_shown = N_samples
    for j in range(N_steps+1):
        t = j * Δt
        if j == N_steps:
            alpha = 1
            N_samples_shown = N_samples
        with torch.no_grad():
            x_ts_test[test] = models_FM[test].step(x_ts_test[test], torch.tensor([t], device=device), torch.tensor([Δt], device=device))

        if j % N_skip == 0:
            im = plot_x(ax[i][1], x_ts_test[test], color=colors[k], marker=".", alpha=alpha)
            k += 1
        alpha = 0.5
        N_samples_shown = N_samples // 4
fig.colorbar(ScalarMappable(cmap=cmap), cax=cax, ticks=np.linspace(0, 1, N_show), label="$t$");
fig.savefig("output/interpolation_R2.pdf", bbox_inches="tight")